# מפגש 12 — פתרון מורה מדויק ומבוקר
תואם מטלת התלמיד: species יעד, flipper_length_mm ו־body_mass_g תכונות; seed=17. קוד זה נועד להרצה על מקור Penguins של הקורס. אין מודל חיזוי.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
url='https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv'
penguins_df=pd.read_csv(url)
raw=penguins_df.copy(deep=True)
missing=penguins_df.isna().sum()
print('Shape',penguins_df.shape,'Missing',missing.to_dict())
assert penguins_df.shape==(344,7)
assert missing['species']==0
assert missing['sex']==11
assert missing['flipper_length_mm']==2 and missing['body_mass_g']==2

## בוחרים שדה יעד ותכונות. לא מוחקים שורות על סמך sex.

In [ ]:
model_df=penguins_df.dropna(subset=['species']).copy()
X=model_df[['flipper_length_mm','body_mass_g']]
y=model_df['species']
print('X and y:',X.shape,y.shape)
assert X.shape==(344,2) and y.shape==(344,)
assert X.index.equals(y.index)
assert 'species' not in X.columns
assert model_df.shape==(344,7)

## פיצול X/y בפקודה אחת; Seed 17 של התלמיד, לא Seed 42 של ההדגמה

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=17,stratify=y)
print('Train/Test:',X_train.shape,X_test.shape)
print('Missing Train:',X_train.isna().sum().to_dict())
print('Missing Test:',X_test.isna().sum().to_dict())
assert X_train.shape==(258,2) and X_test.shape==(86,2)
assert y_train.shape==(258,) and y_test.shape==(86,)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert len(X_train)+len(X_test)==len(X)

## לימוד פרמטרים ב־Train בלבד; יישום בשני הצדדים ללא שינוי מקור

In [ ]:
training_means=X_train.mean()
X_train_ready=X_train.fillna(training_means)
X_test_ready=X_test.fillna(training_means)
print('Training means:',training_means.to_dict())
print('Missing after:',X_train_ready.isna().sum().to_dict(),X_test_ready.isna().sum().to_dict())
assert set(training_means.index)==set(X_train.columns)
assert training_means.notna().all()
assert not X_train_ready.isna().any().any()
assert not X_test_ready.isna().any().any()
assert X_train_ready.index.equals(y_train.index)
assert X_test_ready.index.equals(y_test.index)
assert penguins_df.equals(raw)
assert X_train_ready.shape==X_train.shape and X_test_ready.shape==X_test.shape

## השוואת חציון — רשות בלבד, ללא בחירת שיטה לפי Test

In [ ]:
training_medians=X_train.median()
X_train_median_ready=X_train.fillna(training_medians)
X_test_median_ready=X_test.fillna(training_medians)
print('Medians:',training_medians.to_dict())
assert training_medians.notna().all()
assert not X_train_median_ready.isna().any().any()
assert not X_test_median_ready.isna().any().any()
assert penguins_df.equals(raw)

## בדיקת דוגמת הדליפה העצמאית — ללא קשר לשדות Penguins

In [ ]:
toy_train=pd.Series([10.,20.,None])
toy_test=pd.Series([1000.,None])
fit_mean=toy_train.mean()
wrong_mean=pd.concat([toy_train,toy_test],ignore_index=True).mean()
assert fit_mean==15
assert toy_train.fillna(fit_mean).tolist()==[10.,20.,15.]
assert toy_test.fillna(fit_mean).tolist()==[1000.,15.]
assert np.isclose(wrong_mean,1030/3)
print('Correct mean:',fit_mean,'Leaked all-data mean:',wrong_mean)

## תשובות מורה
אין fit על Test. השלמה על כלל X לפני חלוקה דולפת; ממוצע Test נפרד מפר את המדיניות הנלמדת; יעד חסר אינו תווית אמת. שמירת y ביישור X חיונית. seed לשחזור ולא לאיכות; stratify אינה תחליף לחלוקה מותאמת זמן/קבוצות. לא נבנה מודל ולא נמדד דיוק.